# Baseline: Google Model on the Itajaí-Açu

This notebook does two things:
1. **Trains a local model** using OpenHydroNet with our Itajaí-Açu data
2. **Computes the baseline** (NSE, KGE) on the three critical historical events

---

## NSE and KGE: Why are they the standard metrics in hydrology?

This is one of the most important questions for understanding the field. The answer
reveals what "good forecasting" means in operational hydrology.

### The problem with RMSE and MAE in hydrology

Imagine two basins:
- **Basin A** (Itajaí-Açu): mean of 500 m³/s, peaks of 8,000 m³/s
- **Basin B** (Tijucas/SC): mean of 50 m³/s, peaks of 800 m³/s

A model with RMSE = 200 m³/s is excellent for Basin A (4% of the mean)
but catastrophic for Basin B (400% of the mean). RMSE and MAE are
**scale-dependent** — they do not allow comparing models across basins
or against the literature.

### NSE — Nash-Sutcliffe Efficiency (1970)

```
           Σ(Q_sim - Q_obs)²
NSE = 1 − ─────────────────────
           Σ(Q_obs - mean(Q_obs))²
```

It is the R² between observed and simulated, but with a physical interpretation:

| NSE | Meaning |
|-----|---------|
| 1.0 | Perfect |
| 0.0 | Equivalent to always predicting the historical mean |
| < 0 | Worse than always predicting the mean |

**The denominator is the simplest possible benchmark:** a model that
answers "tomorrow's streamflow will be the historical mean" has NSE = 0.
NSE = 0.7 means your model explains 70% of the variance not explained
by the climatology benchmark.

**Critical limitation of NSE:** the squared error amplifies flood peaks.
A model can have NSE = 0.8 while being excellent on normal days and terrible
during floods — exactly the most important case. NSE is "biased" toward
getting the median streamflow right, not the extremes.

### KGE — Kling-Gupta Efficiency (2009)

Created specifically to solve the NSE problem. It decomposes the error into three components:

```
KGE = 1 − √[(r−1)² + (α−1)² + (β−1)²]

Where:
  r = Pearson correlation  ← does it get the TIMING of peaks right?
  α = std_sim / std_obs    ← does it get the AMPLITUDE of variations right?
  β = mean_sim / mean_obs  ← does it get the total VOLUME right?
```

| KGE | Meaning |
|-----|---------|
| 1.0 | Perfect (r=1, α=1, β=1) |
| −0.41 | Equivalent to the climatology benchmark \* |
| < −0.41 | Worse than always predicting the mean |

\* Knoben et al. (2019): the KGE equivalent of NSE=0 is −0.41, not 0!

**Why is KGE superior for floods?**
The α component penalizes models that get the mean right but underpredict
variability (α < 1). A model that "flattens" flood peaks will have
α << 1 and therefore a low KGE, even if the NSE is reasonable.

### Which one to use?

Current practice in hydrology uses **both**:
- NSE for comparison with the historical literature (backward compatibility)
- KGE as the primary optimization and evaluation metric
- **FHV** (peak flow bias) as a specialized metric for flood events

OpenHydroNet implicitly optimizes KGE via NSELoss/cmalloss and reports both.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os, subprocess
sys.path.insert(0, '../../')
# Vendor path for googlehydrology (installed with pip install -e vendor/flood-forecasting)
# If using the project venv: source .venv/bin/activate
sys.path.insert(0, str(__import__('pathlib').Path('../..') / 'vendor' / 'flood-forecasting'))

from googlehydrology.evaluation import metrics as gh_metrics

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

ROOT         = Path('../..')
CARAVAN_ROOT = ROOT / 'data/processed/Caravan-nc'
CONFIG_DIR   = ROOT / 'configs/training'
MODEL_DIR    = ROOT / 'models/experiments'
FIGDIR       = ROOT / 'reports/figures'
FIGDIR.mkdir(exist_ok=True)

CONFIG_FILE  = CONFIG_DIR / 'mef_lstm_itajai.yml'
BASIN_ID     = 'itajai_83500000'

FLOOD_EVENTS = {
    '1983-11-09': ('Nov/1983', '17.17m'),
    '2008-11-23': ('Nov/2008', '11.78m'),
    '2011-09-08': ('Sep/2011', '10.14m'),
}

print('Environment configured.')
print(f'Config: {CONFIG_FILE}')

## 1. Pre-Training Checks

Before starting training, we verify that all prerequisites
are satisfied. Failing here is much better than failing at epoch 28.

In [ ]:
import yaml
from src.data.caravan_formatter import validate_caravan_structure

checks_ok = True

# 1. Caravan dataset
caravan_checks = validate_caravan_structure(CARAVAN_ROOT)
if not all(caravan_checks.values()):
    print('ERROR: Caravan incomplete. Run first: notebooks/02_preprocessing/01_build_caravan_dataset.ipynb')
    checks_ok = False
else:
    print('✓ Caravan dataset OK')

# 2. Basin list file
basins_file = CONFIG_DIR / 'itajai_basins.txt'
if not basins_file.exists():
    print('ERROR: itajai_basins.txt not found')
    checks_ok = False
else:
    basins = [l.strip() for l in basins_file.read_text().splitlines() if l.strip()]
    print(f'✓ Basin list: {basins}')

# 3. Config YAML
if not CONFIG_FILE.exists():
    print(f'ERROR: config not found at {CONFIG_FILE}')
    checks_ok = False
else:
    with open(CONFIG_FILE) as f:
        cfg = yaml.safe_load(f)
    print(f'✓ Config loaded: {cfg["experiment_name"]}')
    print(f'  model:       {cfg["model"]}')
    print(f'  head:        {cfg["head"]}')
    print(f'  seq_length:  {cfg["seq_length"]} days')
    print(f'  hidden_size: {cfg["hidden_size"]}')
    print(f'  epochs:      {cfg["epochs"]}')
    print(f'  device:      {cfg["device"]}')

# 4. OpenHydroNet installed?
try:
    import googlehydrology
    print(f'✓ googlehydrology imported')
except ImportError:
    print('ERROR: googlehydrology not installed.')
    print('  Run: pip install -e upstream/')
    print('  (clone first: git clone https://github.com/google-research/flood-forecasting.git upstream/)')
    checks_ok = False

print(f'\n{"✓ Ready to train" if checks_ok else "✗ Fix the errors above before continuing"}')

## 2. Training

OpenHydroNet is controlled via command line: `run train --config-file=<path>`.

### How it works internally

1. **Dataset loading**: reads the NetCDF files from Caravan-nc, creates
   sliding windows of `seq_length` days and stacks them into batches.
   Each training sample is a tuple `(hindcast_sequence, forecast_sequence, target)`.

2. **Forward pass**: the MEF-LSTM processes hindcast and forecast in separate LSTMs;
   the final states are combined by the regression head to produce
   the `lead_time` future predictions.

3. **Loss**: MSE computed only on the last `predict_last_n=8` days of the sequence
   (the forecast days). Gradients do not flow back through the historical period.

4. **Validation**: at the end of each epoch, computes NSE and KGE on the validation period.

5. **Output**: weights in `models/experiments/{experiment_name}_{timestamp}/`;
   test results in `test/model_epoch{N}/test_results.zarr`.

In [ ]:
# Build training command
# The 'run' script is installed by googlehydrology's setup.py as an entry point.
train_cmd = f"run train --config-file={CONFIG_FILE.resolve()}"
infer_cmd_template = "run infer --run-dir=<MODEL_RUN_DIR>"

print('Commands to run in the terminal:')
print(f'  conda activate blumenau-flood')
print(f'  cd {ROOT.resolve()}')
print(f'  {train_cmd}')
print()
print('After training completes, replace <MODEL_RUN_DIR> with the created directory:')
print(f'  {infer_cmd_template}')
print()
print('Training will create a directory at:')
print(f'  {MODEL_DIR.resolve()}/{cfg["experiment_name"]}_YYYYMMDD_HHMMSS/')

# ── OPTIONAL: run directly in this notebook ───────────────────────────────────
# Uncomment to train inline (blocks the notebook for ~10-30 min)
#
# import subprocess
# result = subprocess.run(train_cmd.split(), capture_output=True, text=True,
#                         cwd=str(ROOT.resolve()))
# print(result.stdout[-3000:] if result.stdout else '')
# if result.returncode != 0:
#     print('STDERR:', result.stderr[-1000:])

## 3. Loading Results

After training, the framework saves the predictions in Zarr format.
We will automatically load the most recent run.

In [ ]:
import glob, re

def find_latest_run(model_dir: Path, experiment_name: str) -> Path | None:
    """Finds the most recent run directory for the experiment."""
    pattern = str(model_dir / f"{experiment_name}_*")
    candidates = sorted(glob.glob(pattern))
    return Path(candidates[-1]) if candidates else None

def load_test_results(run_dir: Path):
    """
    Loads test results from the most recent epoch.
    
    The framework saves to: test/model_epoch{N}/test_results.zarr
    Variables: streamflow_sim (predicted), streamflow_obs (observed)
    Dimensions: basin × date × time_step (time_step = lead time in days)
    """
    zarr_files = sorted(glob.glob(
        str(run_dir / 'test/model_epoch*/test_results.zarr')
    ))
    if not zarr_files:
        return None, None
    
    # Most recent epoch
    def epoch_num(p):
        m = re.search(r'model_epoch(\d+)', p)
        return int(m.group(1)) if m else 0
    
    latest = max(zarr_files, key=epoch_num)
    epoch = epoch_num(latest)
    ds = xr.open_zarr(latest, consolidated=False)
    return ds, epoch


# Locate run
run_dir = find_latest_run(MODEL_DIR, cfg['experiment_name'])

if run_dir is None:
    print(f'No run found in {MODEL_DIR}.')
    print('Run training in section 2 before continuing.')
    ds_results = None
else:
    print(f'Run found: {run_dir.name}')
    ds_results, epoch = load_test_results(run_dir)
    if ds_results is None:
        print('Test results not found. Run: run infer --run-dir=...')
    else:
        print(f'Results loaded (epoch {epoch})')
        print(ds_results)

## 4. NSE and KGE — Computation by Lead Time

The model generates forecasts for lead times from 1 to 7 days.
Metrics should degrade with the horizon — this is expected and physically correct.

**Interpretation by lead time:**
- **Lead 1:** the model "almost persists" — tomorrow's streamflow is similar to today's.
  A very high NSE here does not necessarily indicate a good model.
- **Lead 3–5:** real test of predictive capacity. The LSTM's memory begins to
  outperform simple persistence.
- **Lead 7:** maximum horizon. NSE > 0.5 here is a strong result.

In [ ]:
def compute_metrics_all_lead_times(ds: xr.Dataset, basin_id: str) -> pd.DataFrame:
    """
    Computes metrics per lead time using googlehydrology.evaluation.metrics.
    NSE, KGE and FHV are computed by the framework — the same implementation used
    in Google's operational evaluation, with no local reimplementation.
    """
    results = []
    lead_times = ds['time_step'].values if 'time_step' in ds.dims else [0]

    for lt in lead_times:
        if 'time_step' in ds.dims:
            sim_da = ds['streamflow_sim'].sel(basin=basin_id, time_step=lt)
        else:
            sim_da = ds['streamflow_sim'].sel(basin=basin_id)

        obs_da = ds['streamflow_obs'].sel(basin=basin_id)

        # Align dates (obs may have a different dimension than sim)
        common_dates = np.intersect1d(sim_da['date'].values, obs_da['date'].values)
        if len(common_dates) < 10:
            continue
        sim_da = sim_da.sel(date=common_dates)
        obs_da = obs_da.sel(date=common_dates)

        try:
            calc = gh_metrics.calculate_metrics(
                obs=obs_da,
                sim=sim_da,
                metrics=['NSE', 'KGE', 'FHV', 'Alpha-NSE', 'Beta-KGE', 'Pearson-r'],
                resolution='1D',
                datetime_coord='date',
            )
        except Exception as e:
            print(f'Lead {lt}: error computing metrics — {e}')
            continue

        results.append({
            'lead_time': int(lt),
            'NSE':      round(calc.get('NSE', float('nan')), 4),
            'KGE':      round(calc.get('KGE', float('nan')), 4),
            'FHV':      round(calc.get('FHV', float('nan')), 1),
            'alpha':    round(calc.get('Alpha-NSE', float('nan')), 4),
            'beta':     round(calc.get('Beta-KGE', float('nan')), 4),
            'r':        round(calc.get('Pearson-r', float('nan')), 4),
        })

    return pd.DataFrame(results).set_index('lead_time')


if ds_results is not None:
    metrics_df = compute_metrics_all_lead_times(ds_results, BASIN_ID)

    print('=== METRICS BY LEAD TIME ===')
    print(metrics_df.to_string())
    print()
    print('KGE interpretation:')
    for lt, row in metrics_df.iterrows():
        kge = row['KGE']
        if kge > 0.7:
            verdict = 'Good (operational)'
        elif kge > 0.5:
            verdict = 'Satisfactory'
        elif kge > -0.41:
            verdict = 'Poor (better than climatology)'
        else:
            verdict = 'Worse than climatology'
        print(f'  Lead {lt}: KGE={kge:.3f} — {verdict}')

    print()
    print('FHV (bias on the top 2% largest events):')
    print('  Positive = model overestimates floods, Negative = underestimates')
    for lt, row in metrics_df.iterrows():
        print(f'  Lead {lt}: FHV = {row["FHV"]:+.1f}%')
else:
    print('No results available yet. Run training first.')


## 5. Plot: Metrics vs. Lead Time

The degradation of metrics with the horizon reveals the model's behavior:
- **Abrupt NSE drop from lead 1→2:** the model has not learned medium-term
  patterns — it relies too heavily on persistence
- **KGE more stable than NSE:** the decomposition into r, α, β distributes the signal;
  the model may degrade in timing (r) while maintaining volume (β)

In [ ]:
if ds_results is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    ax = axes[0]
    ax.plot(metrics_df.index, metrics_df['NSE'], marker='o', color='steelblue', label='NSE')
    ax.plot(metrics_df.index, metrics_df['KGE'], marker='s', color='seagreen', label='KGE')
    ax.axhline(0.7, color='gray', ls='--', lw=1, label='Target (0.70)')
    ax.axhline(0.0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('Lead Time (days)')
    ax.set_ylabel('Score')
    ax.set_title('NSE and KGE vs. Lead Time')
    ax.legend()
    ax.set_ylim(-0.5, 1.05)
    ax.set_xticks(metrics_df.index)
    
    ax2 = axes[1]
    ax2.plot(metrics_df.index, metrics_df['r'],     marker='o', label='r (timing)')
    ax2.plot(metrics_df.index, metrics_df['alpha'], marker='s', label='α (amplitude)')
    ax2.plot(metrics_df.index, metrics_df['beta'],  marker='^', label='β (volume)')
    ax2.axhline(1.0, color='k', ls='--', lw=0.8)
    ax2.set_xlabel('Lead Time (days)')
    ax2.set_ylabel('Component')
    ax2.set_title('KGE Decomposition by Lead Time')
    ax2.legend(fontsize=9)
    ax2.set_xticks(metrics_df.index)
    
    ax3 = axes[2]
    colors = ['salmon' if v > 0 else 'steelblue' for v in metrics_df['FHV']]
    ax3.bar(metrics_df.index, metrics_df['FHV'], color=colors, alpha=0.8)
    ax3.axhline(0, color='k', lw=0.8)
    ax3.axhspan(-20, 20, alpha=0.08, color='green')
    ax3.set_xlabel('Lead Time (days)')
    ax3.set_ylabel('FHV (%)')
    ax3.set_title('Flood Peak Bias (FHV)')
    ax3.set_xticks(metrics_df.index)
    ax3.text(0.98, 0.98, 'Green: acceptable zone (±20%)',
             transform=ax3.transAxes, ha='right', va='top', fontsize=8)
    
    fig.suptitle(f'Baseline — Itajaí-Açu | {cfg["experiment_name"]}', fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGDIR / '18_baseline_metrics_lead_time.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Results not available.')

## 6. Hydrographs at Historical Events

Aggregate metrics hide behavior at the extremes.
This plot reveals **exactly where the model fails** during the largest floods.

Common failure patterns:
- **Timing**: predicted peak 1–2 days early or late
- **Magnitude**: predicted peak well below the actual value (critical problem for alerts)
- **Recession**: model overestimates streamflow during the falling limb of the hydrograph
- **Pre-event**: model does not detect the rapid rise before the peak

Each of these patterns has a different diagnosis:
- Wrong timing → adjust `seq_length` or precipitation features
- Underestimated peak → FHV very negative; consider NSELoss or weighting extremes
- Wrong recession → `hidden_size` too small to capture baseflow memory

In [ ]:
def extract_event_window(
    ds: xr.Dataset,
    basin_id: str,
    peak_date: str,
    lead_time: int = 1,
    window_days: int = 30,
) -> pd.DataFrame | None:
    """
    Extracts observed + simulated data around an event, for a specific lead time.
    """
    peak = pd.Timestamp(peak_date)
    t0   = peak - pd.Timedelta(days=window_days)
    t1   = peak + pd.Timedelta(days=window_days)
    
    try:
        obs = ds['streamflow_obs'].sel(basin=basin_id).to_series()
        if 'time_step' in ds.dims:
            sim = ds['streamflow_sim'].sel(basin=basin_id, time_step=lead_time).to_series()
        else:
            sim = ds['streamflow_sim'].sel(basin=basin_id).to_series()
        
        df = pd.DataFrame({'obs': obs, 'sim': sim})
        df.index = pd.to_datetime(df.index)
        return df.loc[t0:t1]
    except Exception as e:
        print(f'Error extracting event {peak_date}: {e}')
        return None


if ds_results is not None:
    LEAD_TIMES_TO_PLOT = [1, 3, 7]
    WINDOW = 25

    fig, axes = plt.subplots(
        len(FLOOD_EVENTS), len(LEAD_TIMES_TO_PLOT),
        figsize=(5 * len(LEAD_TIMES_TO_PLOT), 4.5 * len(FLOOD_EVENTS)),
        squeeze=False
    )

    for row, (date_str, (label, cota_str)) in enumerate(FLOOD_EVENTS.items()):
        for col, lt in enumerate(LEAD_TIMES_TO_PLOT):
            ax = axes[row][col]
            evt = extract_event_window(ds_results, BASIN_ID, date_str, lead_time=lt, window_days=WINDOW)

            if evt is None or evt.dropna().empty:
                ax.text(0.5, 0.5, 'Data\nunavailable', ha='center', va='center',
                        transform=ax.transAxes)
                continue

            ax.plot(evt.index, evt['obs'], color='k', lw=1.5, label='Observed')
            ax.plot(evt.index, evt['sim'], color='steelblue', lw=1.5, ls='--',
                    label=f'Predicted (L{lt})')

            # Observed peak
            peak_dt = evt['obs'].idxmax()
            peak_v  = evt['obs'].max()
            ax.scatter([peak_dt], [peak_v], c='crimson', s=50, zorder=6)
            ax.axvline(pd.Timestamp(date_str), color='crimson', lw=1, ls=':', alpha=0.5)

            # Event metrics
            if lt in metrics_df.index:
                nse_evt = metrics_df.loc[lt, 'NSE']
                kge_evt = metrics_df.loc[lt, 'KGE']
                ax.text(0.02, 0.97,
                        f'NSE={nse_evt:.2f}\nKGE={kge_evt:.2f}',
                        transform=ax.transAxes, va='top', fontsize=8,
                        bbox=dict(facecolor='white', alpha=0.8))

            if col == 0:
                ax.set_title(f'{label} (gauge {cota_str})', fontsize=10)
                ax.set_ylabel('Streamflow (m³/s)')
            else:
                ax.set_title(f'Lead {lt} days', fontsize=10)

            ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
            ax.legend(fontsize=7)

    fig.suptitle('Hydrographs at Historical Events — Predicted vs. Observed',
                 fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGDIR / '19_baseline_hydrographs.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run training and inference first.')

## 7. Flow Duration Curve: The Full Distribution

The FDC (Flow Duration Curve) shows model performance across **the full distribution**,
not just during extreme events.

Gaps between the curves reveal:
- **Head (Q > P10):** bias at flood peaks — coincides with FHV
- **Middle (P20–P70):** average regime — where NSE is most sensitive
- **Tail (Q < P70):** low flows — where FLV measures performance

In [ ]:
if ds_results is not None:
    lead_for_fdc = 1  # show FDC for lead 1 day (best-case scenario)
    
    obs_all = ds_results['streamflow_obs'].sel(basin=BASIN_ID).values.flatten()
    if 'time_step' in ds_results.dims:
        sim_all = ds_results['streamflow_sim'].sel(basin=BASIN_ID, time_step=lead_for_fdc).values
    else:
        sim_all = ds_results['streamflow_sim'].sel(basin=BASIN_ID).values.flatten()
    
    mask = (~np.isnan(obs_all)) & (~np.isnan(sim_all[:len(obs_all)]))
    obs_v = obs_all[mask]
    sim_v = sim_all[:len(obs_all)][mask]
    
    # Exceedance probabilities
    ep_obs = np.arange(1, len(obs_v)+1) / (len(obs_v)+1)
    ep_sim = np.arange(1, len(sim_v)+1) / (len(sim_v)+1)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # FDC on log scale
    axes[0].semilogy(ep_obs * 100, np.sort(obs_v)[::-1],
                     color='k', lw=2, label='Observed')
    axes[0].semilogy(ep_sim * 100, np.sort(sim_v)[::-1],
                     color='steelblue', lw=1.5, ls='--', label=f'Simulated (Lead {lead_for_fdc}d)')
    axes[0].set_xlabel('Exceedance Probability (%)')
    axes[0].set_ylabel('Streamflow (m³/s) — log scale')
    axes[0].set_title('Flow Duration Curve')
    axes[0].legend()
    axes[0].axvline(2, color='salmon', lw=1, ls=':', label='FHV region (2%)')
    axes[0].axvline(70, color='gray', lw=1, ls=':')
    
    # Scatter obs vs. sim (log scale)
    lim_max = max(obs_v.max(), sim_v.max()) * 1.05
    axes[1].scatter(obs_v, sim_v, s=2, alpha=0.3, color='steelblue')
    axes[1].plot([0, lim_max], [0, lim_max], 'k--', lw=1, label='1:1')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Observed (m³/s)')
    axes[1].set_ylabel('Simulated (m³/s)')
    axes[1].set_title(f'Scatter Obs vs. Sim — Lead {lead_for_fdc}d')
    
    fig.tight_layout()
    fig.savefig(FIGDIR / '20_baseline_fdc.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run training and inference first.')

## 8. Diagnosis and Next Steps

Based on the metrics and hydrographs above, we fill in the baseline diagnosis.

In [ ]:
if ds_results is not None and not metrics_df.empty:
    lt1 = metrics_df.loc[1] if 1 in metrics_df.index else metrics_df.iloc[0]
    lt7 = metrics_df.loc[7] if 7 in metrics_df.index else metrics_df.iloc[-1]
    
    print('=' * 55)
    print('  BASELINE DIAGNOSIS')
    print('=' * 55)
    print(f'  NSE Lead-1:  {lt1["NSE"]:.3f}   KGE Lead-1:  {lt1["KGE"]:.3f}')
    print(f'  NSE Lead-7:  {lt7["NSE"]:.3f}   KGE Lead-7:  {lt7["KGE"]:.3f}')
    print(f'  FHV Lead-1:  {lt1["FHV"]:+.1f}%  FHV Lead-7:  {lt7["FHV"]:+.1f}%')
    print(f'  Lead-1 decomposition: r={lt1["r"]:.3f}, α={lt1["alpha"]:.3f}, β={lt1["beta"]:.3f}')
    print()
    
    # Automatic diagnosis
    issues = []
    if lt1['alpha'] < 0.8:
        issues.append('α < 0.8: model underestimates variability → consider NSELoss or increasing hidden_size')
    if abs(lt1['beta'] - 1) > 0.2:
        issues.append(f'β = {lt1["beta"]:.2f}: systematic volume bias → check normalization')
    if lt1['r'] < 0.8:
        issues.append('r < 0.8: poor timing → consider increasing seq_length or adding precipitation features')
    if abs(lt1['FHV']) > 30:
        issues.append(f'FHV = {lt1["FHV"]:+.0f}%: large peak bias → model may fail in flood alerts')
    if lt1['KGE'] > 0.7:
        issues.append('KGE Lead-1 > 0.7: solid baseline. Next step: fine-tuning to improve Lead-7.')
    elif lt1['KGE'] > 0.5:
        issues.append('KGE Lead-1 between 0.5–0.7: functional model but below the operational target of 0.7.')
    else:
        issues.append('KGE Lead-1 < 0.5: model below expectations. Check data pipeline before fine-tuning.')
    
    print('  Diagnosis:')
    for issue in issues:
        print(f'  → {issue}')
    
    print()
    print('  Suggested next steps:')
    print('  1. notebooks/03_training/02_finetune.ipynb — fine-tune on extreme events')
    print('  2. Add ERA5-Land as dynamic features (temperature, evapotranspiration)')
    print('  3. Extract HydroAtlas and ERA5 attributes to improve the static embedding')
    print('  4. Ablation experiment: seq_length (14 vs 30 vs 60)')
    print('=' * 55)
else:
    print('Run training and sections 3–5 before the diagnosis.')